# AquaSense: Machine Learning-Based Water Potability Assessment

**Odoi Stephen | Makerere University | MSC 7103 Machine Learning**

This notebook reconstructs the project analysis from the supplied CSV. It compares logistic regression, random forest and gradient boosting, selects a training-only threshold for the recorded non-potable label, and evaluates one held-out test set. The results describe the dataset labels; they do not certify drinking-water safety.

Place `water_potability.csv` in the same folder as this notebook. In Google Colab, upload the CSV to the current session using the folder icon before running all cells. The first execution may take a few minutes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import make_scorer, recall_score, precision_score, accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve

DATA = Path('water_potability.csv')
if not DATA.exists():
    raise FileNotFoundError('Upload water_potability.csv to the notebook working folder first.')
data = pd.read_csv(DATA)
X = data.drop(columns='Potability')
y = data['Potability'].astype(int)
print('Dataset shape:', data.shape)
print('Class counts (0 = non-potable, 1 = potable):\n', y.value_counts().sort_index())
print('Missing values:\n', data.isna().sum())
print('Rows with any missing predictor:', X.isna().any(axis=1).sum())


## 1. Hold-out split and preprocessing

The split is stratified, with 80% training and 20% testing. Imputation and scaling live inside each pipeline so they are fitted on each training fold only.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def pipeline(model):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), model)

models = {
    'Logistic regression': pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    'Random forest': pipeline(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    'Gradient boosting': pipeline(GradientBoostingClassifier(n_estimators=100, random_state=42)),
}
print('Training rows:', len(X_train), '| Test rows:', len(X_test))


## 2. Compare models on the training data

Scores in this table use each model's default class decision rule. Non-potable is the positive class for its recall and precision. ROC-AUC is calculated from the potable-class probability, which produces the same AUC as the non-potable score with the class coding reversed.


In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'nonpotable_recall': make_scorer(recall_score, pos_label=0),
    'nonpotable_precision': make_scorer(precision_score, pos_label=0, zero_division=0),
    'potable_recall': make_scorer(recall_score, pos_label=1),
    'roc_auc': 'roc_auc',
}
rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, scoring=scoring, cv=cv, n_jobs=1)
    rows.append({'Model': name, **{metric: scores['test_' + metric].mean() for metric in scoring}})
comparison = pd.DataFrame(rows).set_index('Model')
print(comparison.round(3).to_string())


## 3. Choose the non-potable threshold

The random forest achieved the highest cross-validation ROC-AUC. Use pooled out-of-fold probabilities from training folds to find the highest tested threshold meeting at least 95% non-potable recall. The test set is not used to choose this threshold.


In [ ]:
forest = models['Random forest']
# Column 0 corresponds to original label 0: non-potable.
train_score_0 = cross_val_predict(forest, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1)[:, 0]
threshold_grid = np.round(np.arange(0, 1.0001, 0.005), 3)
valid = [t for t in threshold_grid if recall_score(y_train, np.where(train_score_0 >= t, 0, 1), pos_label=0) >= 0.95]
threshold = max(valid)
print('Selected non-potable score threshold:', threshold)
threshold_rows=[]
for t in sorted(set([0.2, 0.3, 0.4, threshold, 0.5, 0.6, 0.7])):
    prediction=np.where(train_score_0 >= t, 0, 1)
    matrix=confusion_matrix(y_train, prediction, labels=[0,1])
    threshold_rows.append({'threshold': t, 'recall_0':recall_score(y_train,prediction,pos_label=0),
                           'recall_1':recall_score(y_train,prediction,pos_label=1),
                           'precision_0':precision_score(y_train,prediction,pos_label=0,zero_division=0),
                           'nonpotable_missed':matrix[0,1], 'potable_flagged':matrix[1,0]})
threshold_table=pd.DataFrame(threshold_rows)
print(threshold_table.round(4).to_string(index=False))


## 4. Refit the forest and evaluate the test set

Here, a false negative is a recorded non-potable case predicted potable. A false positive is a potable-labelled case flagged non-potable.


In [ ]:
forest.fit(X_train, y_train)
test_score_0=forest.predict_proba(X_test)[:,0]
test_pred=np.where(test_score_0 >= threshold,0,1)
cm=confusion_matrix(y_test,test_pred,labels=[0,1])
print('Rows = recorded labels 0/1; columns = predicted labels 0/1:\n',cm)
print('Accuracy:',round(accuracy_score(y_test,test_pred),3))
print('Non-potable recall:',round(recall_score(y_test,test_pred,pos_label=0),3))
print('Non-potable precision:',round(precision_score(y_test,test_pred,pos_label=0),3))
print('Non-potable F1:',round(f1_score(y_test,test_pred,pos_label=0),3))
print('Potable recall:',round(recall_score(y_test,test_pred,pos_label=1),3))
print('Test ROC-AUC:',round(roc_auc_score((y_test==0).astype(int),test_score_0),3))


## 5. ROC curve and feature importance

Impurity importance summarizes the fitted forest and is not a causal effect.


In [ ]:
fpr,tpr,_=roc_curve((y_test==0).astype(int),test_score_0)
auc=roc_auc_score((y_test==0).astype(int),test_score_0)
fig,ax=plt.subplots(figsize=(6,5))
ax.plot(fpr,tpr,label=f'Random forest (AUC = {auc:.3f})')
ax.plot([0,1],[0,1],'--',color='grey',label='Chance')
ax.scatter(cm[1,0]/sum(cm[1]),cm[0,0]/sum(cm[0]),color='darkorange',label=f'Threshold {threshold:.3f}')
ax.set(xlabel='False positive rate',ylabel='Non-potable recall',title='Held-out test ROC curve')
ax.legend(); ax.grid(alpha=.25); plt.show()
importance=pd.Series(forest[-1].feature_importances_,index=X.columns).sort_values(ascending=False)
print('Feature importance:\n',importance.round(4).to_string())
importance.sort_values().plot.barh(figsize=(7,5),title='Random forest feature importance')
plt.xlabel('Relative impurity-based importance'); plt.tight_layout(); plt.show()


## 6. Example record and model bundle

Record 664 is one illustrative test record. The saved bundle includes the fitted pipeline and the selected threshold. The model is for research only and must not be used to certify drinking-water safety.


In [ ]:
if 664 in X_test.index:
    i=list(X_test.index).index(664)
    print('Record 664:', {'nonpotable_score':round(float(test_score_0[i]),4),
                            'threshold':threshold,'predicted_label':int(test_pred[i]),
                            'recorded_label':int(y_test.iloc[i])})
joblib.dump({'pipeline':forest,'nonpotable_threshold':threshold,
             'feature_names':list(X.columns),'nonpotable_label':0},
            'AquaSense_random_forest_v1.joblib')
print('Saved AquaSense_random_forest_v1.joblib')
